# Top 5 Movies of Each Category (Genre)
MovieLens 100k: movies are ranked by their average rating. 

### Load imported libraries

In [1]:
# importing libraries
# import pandas for working with tables (DataFrames)
import pandas as pd

### Load the datasets

In [2]:
#Read the movie Dataset
# column names of the MovieLens u.item file
i_cols = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
    'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
    'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

# read the pipe separated movie file using those column names
items = pd.read_csv('ml-100k/u.item', sep = '|', names = i_cols, encoding = 'latin-1')

In [3]:
#read the ratings dataset
# column names of the MovieLens u.data file
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']

# read the tab separated ratings file into a DataFrame
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols,encoding='latin-1')

### Average rating and number of ratings of every movie
These are kept as separate lists (Series), so no column is added to `items` or `ratings`.

In [4]:
# average rating of every movie (index = movie id)
avg_rating = ratings.groupby('movie_id')['rating'].mean()

# number of ratings of every movie (index = movie id)
rating_count = ratings.groupby('movie_id')['rating'].count()

# title of every movie (index = movie id)
movie_title = items.set_index('movie id')['movie title']

### Keep only movies with enough ratings
A movie rated 5 by a single user should not beat a movie rated 4.5 by hundreds of users.

In [5]:
# minimum number of ratings a movie must have to be ranked (change this number if you want)
min_ratings = 50

# movie ids that have at least that many ratings
qualified_ids = rating_count[rating_count >= min_ratings].index

In [6]:
# number of movies that qualify
len(qualified_ids)

603

### Top 5 movies for each category

In [7]:
# the 18 genre columns (a movie can belong to more than one genre)
genres = ['Action', 'Adventure',
    'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
    'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

In [8]:
# empty dictionary to store the top 5 movie ids of every genre
top_five_ids = {}

# loop over every genre
for genre in genres:
    # movie ids of this genre (genre column = 1)
    genre_ids = items[items[genre] == 1]['movie id']

    # keep only the movies that have enough ratings
    genre_ids = genre_ids[genre_ids.isin(qualified_ids)]

    # sort by average rating (highest first); if tied, the movie with more ratings comes first
    genre_sorted = sorted(genre_ids, key = lambda m: (-avg_rating[m], -rating_count[m]))

    # keep the first 5 movie ids of this genre
    top_five_ids[genre] = genre_sorted[:5]

### Show the result

In [9]:
# loop over every genre
for genre in genres:
    # print the genre name
    print(genre)

    # loop over its top 5 movies with the rank number
    for rank, m in enumerate(top_five_ids[genre], start = 1):
        # print rank, title, average rating and number of ratings
        print(rank, movie_title[m], '| average rating:', round(avg_rating[m], 2), '| ratings:', rating_count[m])

    # print an empty line between genres
    print()

Action
1 Star Wars (1977) | average rating: 4.36 | ratings: 583
2 Godfather, The (1972) | average rating: 4.28 | ratings: 413
3 Raiders of the Lost Ark (1981) | average rating: 4.25 | ratings: 420
4 Titanic (1997) | average rating: 4.25 | ratings: 350
5 Empire Strikes Back, The (1980) | average rating: 4.2 | ratings: 367

Adventure
1 Star Wars (1977) | average rating: 4.36 | ratings: 583
2 Raiders of the Lost Ark (1981) | average rating: 4.25 | ratings: 420
3 Lawrence of Arabia (1962) | average rating: 4.23 | ratings: 173
4 Empire Strikes Back, The (1980) | average rating: 4.2 | ratings: 367
5 African Queen, The (1951) | average rating: 4.18 | ratings: 152

Animation
1 Close Shave, A (1995) | average rating: 4.49 | ratings: 112
2 Wrong Trousers, The (1993) | average rating: 4.47 | ratings: 118
3 Wallace & Gromit: The Best of Aardman Animation (1996) | average rating: 4.45 | ratings: 67
4 Grand Day Out, A (1992) | average rating: 4.11 | ratings: 66
5 Toy Story (1995) | average rating: 3

In [10]:
# one table: the genres are the columns and the rows are rank 1 to 5 (movie titles only)
top_five_table = pd.DataFrame({genre: pd.Series([movie_title[m] for m in top_five_ids[genre]], index = range(1, len(top_five_ids[genre]) + 1)) for genre in genres})

# show the table
top_five_table

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,Star Wars (1977),Star Wars (1977),"Close Shave, A (1995)","Wizard of Oz, The (1939)","Close Shave, A (1995)","Usual Suspects, The (1995)",Hoop Dreams (1994),Schindler's List (1993),E.T. the Extra-Terrestrial (1982),"Manchurian Candidate, The (1962)",Psycho (1960),"Wizard of Oz, The (1939)",Rear Window (1954),Casablanca (1942),Star Wars (1977),"Close Shave, A (1995)",Schindler's List (1993),High Noon (1952)
2,"Godfather, The (1972)",Raiders of the Lost Ark (1981),"Wrong Trousers, The (1993)",Babe (1995),"Wrong Trousers, The (1993)","Godfather, The (1972)","Celluloid Closet, The (1995)",Casablanca (1942),Heavenly Creatures (1994),"Maltese Falcon, The (1941)",Alien (1979),Singin' in the Rain (1952),"Third Man, The (1949)",Star Wars (1977),Dr. Strangelove or: How I Learned to Stop Worr...,Rear Window (1954),Casablanca (1942),Butch Cassidy and the Sundance Kid (1969)
3,Raiders of the Lost Ark (1981),Lawrence of Arabia (1962),Wallace & Gromit: The Best of Aardman Animatio...,Toy Story (1995),North by Northwest (1959),"Godfather: Part II, The (1974)",Crumb (1994),"Shawshank Redemption, The (1994)","20,000 Leagues Under the Sea (1954)",Sunset Blvd. (1950),Young Frankenstein (1974),This Is Spinal Tap (1984),Vertigo (1958),Titanic (1997),"Empire Strikes Back, The (1980)","Usual Suspects, The (1995)",Star Wars (1977),"Magnificent Seven, The (1954)"
4,Titanic (1997),"Empire Strikes Back, The (1980)","Grand Day Out, A (1992)",E.T. the Extra-Terrestrial (1982),As Good As It Gets (1997),L.A. Confidential (1997),Looking for Richard (1996),12 Angry Men (1957),Jumanji (1995),L.A. Confidential (1997),"Shining, The (1980)","Blues Brothers, The (1980)","Maltese Falcon, The (1941)","Empire Strikes Back, The (1980)",Blade Runner (1982),"Third Man, The (1949)",Dr. Strangelove or: How I Learned to Stop Worr...,Unforgiven (1992)
5,"Empire Strikes Back, The (1980)","African Queen, The (1951)",Toy Story (1995),Aladdin (1992),Cinema Paradiso (1988),Fargo (1996),Koyaanisqatsi (1983),Citizen Kane (1941),"Mask, The (1994)",Blade Runner (1982),"Birds, The (1963)",My Fair Lady (1964),Amadeus (1984),"African Queen, The (1951)",Alien (1979),"Silence of the Lambs, The (1991)",Lawrence of Arabia (1962),"Good, The Bad and The Ugly, The (1966)"


**Notes**
- A movie with more than one genre (for example Comedy and Romance) appears in each of its genres.
- Some genres, such as Film-Noir or Documentary, may have fewer than 5 movies with at least `min_ratings` ratings. Lower `min_ratings` to see more.
- The `unknown` genre is not included.